In [ ]:
import math
import numpy as np
import random
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
class Value:
    def __init__(self,data,_children=(),_op = '',label=''):
        self.data = data
        self.grad = 0.0
        self._backward = lambda:None 
        self._prev = _children
        self._op = _op
        self.label = label
    def __repr__(self):
        return f"Value(data = {self.data})"
    
    def __add__(self,other):
        other = other if isinstance(other,Value) else Value(other)   #used for operations including an int and a Value obj like Value(4)+1
        out = Value(self.data + other.data, {self,other}, '+')
        
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad 
        out._backward = _backward
        return out 

    def __radd__(self, other):
        return self + other

    def __mul__(self,other):  # works for Value * int but not for int * value as it is value.__mul__(int) thus rmul is used 
        other = other if isinstance(other,Value) else Value(other)
        out = Value(self.data * other.data, {self,other}, '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad 
        out._backward = _backward
        return out

    def __pow__(self,other):
        assert isinstance(other,(int, float)) , "pow supported types : int and float only"
        out = Value(self.data**other, (self,), f'**{other}')

        def _backward():
            self.grad+= other * (self.data ** (other-1)) *out.grad
        out._backward = _backward
        return out
    
    def __rmul__(self,other): #works for swapping the Value with int and thus perform Value.__mul__(int) instead of int.__mul__(val)
        return self * other 

    def __truediv__(self,other):
        return self * other**-1   # (a/b = a * b**-1) 

    def __neg__(self):
        return self * -1

    def __sub__(self,other):
        return self + (-other)
    
    def __rsub__(self,other):
        return other + (-self)
        
    def tanh(self):
        x = self.data
        t = (math.exp(2*x) -1)/(math.exp(2*x) + 1)
        out = Value(t, {self,}, 'tanh')
        
        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        
        return out

    def exp(self):  # for e^x
        x= self.data
        out = Value(math.exp(x), (self,) , 'exp')

        def _backward():
            self.grad += out.data * out.grad
        out._backward = _backward
        return out
    
    def log(self):
        x = self.data
        out = Value(math.log(x), (self,), 'log')

        def _backward():
            self.grad += (1/x) * out.grad
        out._backward = _backward
        return out
    
    def sigmoid(self):
        x = self.data
        s = 1/(1 + math.exp(-x))
        out = Value(s, (self,), 'sigmoid')

        def _backward():
            self.grad += s * (1-s) * out.grad
        out._backward = _backward
        return out
    
    def relu(self):
        x = self.data
        out = Value(x if x>0 else 0, (self,), 'relu')

        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward
        return out
    
    

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        self.grad = 1.0
        for node in reversed(topo):
            node._backward()


In [ ]:
from graphviz import Digraph
#toposort for proper u to v relation where u comes before v depicting proper computation of the gradients
def trace(root):
    #build the node and edges of the graph
    nodes,edges = set(),set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child,v))
                build(child)
    build(root)
    return nodes,edges

def draw_dot(root):
    dot = Digraph(format = 'svg',graph_attr={'rankdir':'LR'})
    nodes,edges = trace(root)
    for n in nodes:
        uid = str(id(n))
        #rectangular node for data
        dot.node(name= uid , label = "{%s | data %.4f | grad %.4f}" % (n.label ,n.data,n.grad,), shape= 'record')
        if n._op:
            #value after operation node
            dot.node(name =uid+ n._op, label = n._op)
            dot.edge(uid+ n._op,uid)
    for n1,n2 in edges:
        dot.edge(str(id(n1)),str(id(n2)) + n2._op)
    return dot

In [ ]:
#inputs x1,x2
x1 = Value(2.0 , label = 'x1')
x2 = Value(0.0 , label = 'x2')
#weights w1,w2
w1 = Value(-3.0, label = 'w1')
w2 = Value(1.0, label = 'w2')
#bias of the neuron
b = Value(6.8813735870195432, label = 'b')
#x1w1 + x2w2 +b
x1w1 = x1*w1; x1w1.label = 'x1w1'
x2w2 = x2*w2; x2w2.label = 'x2w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1w1 + x2w2'
n = x1w1x2w2 + b; n.label = 'n'
o = n.tanh(); o.label = 'o'


In [ ]:
o.backward()
draw_dot(o)

In [ ]:
#inputs x1,x2
x1 = Value(2.0 , label = 'x1')
x2 = Value(0.0 , label = 'x2')
#weights w1,w2
w1 = Value(-3.0, label = 'w1')
w2 = Value(1.0, label = 'w2')
#bias of the neuron
b = Value(6.8813735870195432, label = 'b')
#x1w1 + x2w2 +b
x1w1 = x1*w1; x1w1.label = 'x1w1'
x2w2 = x2*w2; x2w2.label = 'x2w2'
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'x1w1 + x2w2'
n = x1w1x2w2 + b; n.label = 'n'
#expanding tanh   -> ((e**2x) -1) / ((e**2x) +1)
e = (2*n).exp()
o = (e-1)/(e+1)
#--------------
o.label = 'o'
o.backward()


In [ ]:
class Module:
    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0.0

    def parameters(self):
        return []

def apply_activation(x,activation):
    if activation == 'tanh':
        return x.tanh()
    elif activation == 'relu':
        return x.relu()
    elif activation == 'sigmoid':
        return x.sigmoid()
    elif activation in ['linear',None]:
        return x
    else:
        raise ValueError(f"Unsupported activation function: {activation}")
    

class Neuron(Module):
    def __init__(self,nin,activation='tanh'):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1,1))
        self.activation = activation

    #forward pass
    def __call__(self,x):
        act = sum((wi*xi for wi , xi in zip(self.w, x)),self.b)  # acivation funciton 
        out = apply_activation(act, self.activation)
        return out

    def parameters(self):
        return self.w + [self.b]
    
class Layer(Module):
    def __init__(self,nin,nout,activation='tanh'):
        self.neurons = [Neuron(nin, activation) for _ in range(nout)]

    def __call__(self,x):
        outs = [n(x) for n in self.neurons]
        return outs[0] if len(outs) == 1 else outs
    
    def parameters(self):
        params = []
        for neuron in self.neurons:
            params.extend(neuron.parameters())
        return params

class MLP(Module):
    def __init__(self,nin,nouts,activation='tanh', output_activation="linear"):
        sz = [nin] +nouts
        self.layers =[]
        for i in range(len(nouts)):
            is_output_layer = (i == len(nouts) - 1)
            layer_activation = output_activation if is_output_layer else activation
            self.layers.append(Layer(sz[i],sz[i+1],activation=layer_activation))

    def __call__(self,x):
        for layer in self.layers:
            x = layer(x)
        return x
    
    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]
        
    

In [ ]:
x = [2.0,3.0,-1.0]
n = MLP(3,[4,4,1])

xs = [[2.0,3.0,-1.0],
      [3.0,-1.0,0.5],
      [0.5,1.0,1.0],
      [1.0,1.0,-1.0]]
ys = [1.0,-1.0,-1.0,1.0] # desired output for the above inputs


In [ ]:
for k in range(25):
    #forward pass
    ypred = [n(x) for x in xs]
    loss = sum((yout - ygt)**2 for ygt,yout in zip(ys,ypred))
    
    #backward pass
    for p in n.parameters():
        p.grad = 0.0  # reset the gradients to zero before each update(.zero_grad)
    loss.backward()

    #update the weights and biases using gradient descent
    for p in n.parameters():
        p.data -= 0.05 * p.grad

    print(k, loss.data)

In [ ]:
ypred

In [ ]:
def binary_cross_entropy(y_pred,y_true):
    y_true = y_true if isinstance(y_true,Value) else Value(y_true)
    eps = 1e-12
    y_pred = y_pred * (1-2 * eps)+eps
    return -(y_true * y_pred.log() + (1-y_true) * (1-y_pred).log())

class BinaryMLPClassifier(Module):
    def __init__(self,nin,nouts,activation='tanh'):
        self.model = MLP(
            nin,
            nouts + [1],
            activation=activation,
            output_activation='linear',
        )
    def logit(self,x):
        return self.model(x)
    
    def predict_proba(self,x):
        return self.logit(x).sigmoid()
    
    def predict(self,x,threshold=0.5):
        return int(self.predict_proba(x).data >= threshold)
    
    def loss(self,xs,ys):
        losses = []
        for x,y in zip(xs,ys):
            prob = self.predict_proba(x)
            losses.append(binary_cross_entropy(prob,y))
        return sum(losses)/len(losses)
    
    def accuracy(self,xs,ys):
        correct = 0 
        for x,y in zip(xs,ys):
            correct += self.predict(x) == y
        return correct/len(ys)
    
    def parameters(self):
        return self.model.parameters() 



In [ ]:
class Optimizer:
    def __init__(self,params,lr = 0.01):
        self.params = list(params)
        self.lr = lr 
    
    def zero_grad(self):
        for p in self.params:
            p.grad = 0.0 
    
    def step(self):
        raise NotImplementedError("Optimizer step method must be implemented by subclasses")
    
class SGD(Optimizer):
    def step(self):
        for p in self.params:
            p.data -= self.lr * p.grad

class Momentum(Optimizer):
    def __init__(self,params,lr=0.01,momentum = 0.9):
        super().__init__(params,lr)
        self.momentum = momentum
        self.velocity = {id(p): 0.0 for p in self.params}
    
    def step(self):
        for p in self.params:
            key = id(p)
            self.velocity[key] = self.momentum * self.velocity[key] + p.grad
            p.data -= self.lr * self.velocity[key]

class RMSProp(Optimizer):
    def __init__(self,params,lr = 0.01, beta = 0.9 , eps = 1e-8):
        super().__init__(params,lr)
        self.beta = beta
        self.eps = eps
        self.square_avg = {id(p): 0.0 for p in self.params}

    def step(self):
        for p in self.params:
            key = id(p)
            self.square_avg[key] = self.beta * self.square_avg[key] + (1-self.beta) * (p.grad ** 2)
            p.data -= self.lr * p.grad / (math.sqrt(self.square_avg[key]) + self.eps)
            
class Adam(Optimizer):
    def __init__(self, params, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8):
        super().__init__(params, lr)
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.t = 0
        self.m = {id(p): 0.0 for p in self.params}
        self.v = {id(p): 0.0 for p in self.params}

    def step(self):
        self.t += 1

        for p in self.params:
            key = id(p)

            self.m[key] = self.beta1 * self.m[key] + (1 - self.beta1) * p.grad
            self.v[key] = self.beta2 * self.v[key] + (1 - self.beta2) * p.grad**2

            m_hat = self.m[key] / (1 - self.beta1**self.t)
            v_hat = self.v[key] / (1 - self.beta2**self.t)

            p.data -= self.lr * m_hat / (v_hat**0.5 + self.eps)

In [ ]:
xor_xs = [
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
]

xor_ys = [0.0, 1.0, 1.0, 0.0]

In [ ]:
random.seed(42)

model = BinaryMLPClassifier(2, [4, 4], activation="tanh")
optimizer = Adam(model.parameters(), lr=0.03)

for epoch in range(250):
    loss = model.loss(xor_xs, xor_ys)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 25 == 0:
        acc = model.accuracy(xor_xs, xor_ys)
        print(epoch, loss.data, acc)

print("probabilities:", [round(model.predict_proba(x).data, 3) for x in xor_xs])
print("predictions:", [model.predict(x) for x in xor_xs])


In [ ]:
def train_classifier(model, optimizer, xs, ys, epochs=250):
    history = {
        "loss": [],
        "accuracy": [],
    }

    for epoch in range(epochs):
        loss = model.loss(xs, ys)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        history["loss"].append(loss.data)
        history["accuracy"].append(model.accuracy(xs, ys))

    return history

In [ ]:
def plot_decision_boundary(model, xs, ys, title="Decision Boundary"):
    x_min, x_max = -0.5, 1.5
    y_min, y_max = -0.5, 1.5
    step = 0.02

    grid_x = np.arange(x_min, x_max, step)
    grid_y = np.arange(y_min, y_max, step)

    zz = []

    for y in grid_y:
        row = []
        for x in grid_x:
            prob = model.predict_proba([x, y]).data
            row.append(prob)
        zz.append(row)

    plt.figure(figsize=(5, 5))
    plt.contourf(grid_x, grid_y, zz, levels=30, cmap="coolwarm", alpha=0.75)

    xs_np = np.array(xs)
    plt.scatter(
        xs_np[:, 0],
        xs_np[:, 1],
        c=ys,
        cmap="coolwarm",
        edgecolors="black",
        s=80,
    )

    plt.xlabel("x1")
    plt.ylabel("x2")
    plt.title(title)
    plt.colorbar(label="Predicted probability")
    plt.show()

plot_decision_boundary(model, xor_xs, xor_ys, title="XOR Decision Boundary")

In [ ]:
optimizer_builders = {
    "SGD": lambda params: SGD(params, lr=0.1),
    "Momentum": lambda params: Momentum(params, lr=0.05, momentum=0.9),
    "RMSProp": lambda params: RMSProp(params, lr=0.02),
    "Adam": lambda params: Adam(params, lr=0.03),
}

xor_histories = {}

for name, build_optimizer in optimizer_builders.items():
    random.seed(42)

    model = BinaryMLPClassifier(2, [4, 4], activation="tanh")
    optimizer = build_optimizer(model.parameters())

    history = train_classifier(model, optimizer, xor_xs, xor_ys, epochs=250)
    xor_histories[name] = history

    print(
        name,
        "final loss:",
        round(history["loss"][-1], 4),
        "final accuracy:",
        history["accuracy"][-1],
    )

In [ ]:
plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
for name, history in xor_histories.items():
    plt.plot(history["loss"], label=name)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("XOR Loss")
plt.legend()

plt.subplot(1, 2, 2)
for name, history in xor_histories.items():
    plt.plot(history["accuracy"], label=name)

plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("XOR Accuracy")
plt.legend()

plt.tight_layout()
plt.show()

### Banknote Classifier 

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("../data/banknote_authentication.txt", header=None)
df.columns = ["variance", "skewness", "curtosis", "entropy", "class"]

# Remove duplicate feature rows
df_clean = df.drop_duplicates(
    subset=["variance", "skewness", "curtosis", "entropy"]
).reset_index(drop=True)

feature_cols = df_clean.columns[:-1]
label_col = df_clean.columns[-1]

banknote_xs = df_clean[feature_cols].values.tolist()
banknote_ys = df_clean[label_col].astype(float).values.tolist()

print("original rows:", len(df))
print("rows after dedup:", len(df_clean))
print(df_clean["class"].value_counts())


def train_test_split(xs, ys, test_size=0.2, seed=42):
    random.seed(seed)

    indices = list(range(len(xs)))
    random.shuffle(indices)

    split = int(len(xs) * (1 - test_size))

    train_indices = indices[:split]
    test_indices = indices[split:]

    train_xs = [xs[i] for i in train_indices]
    train_ys = [ys[i] for i in train_indices]

    test_xs = [xs[i] for i in test_indices]
    test_ys = [ys[i] for i in test_indices]

    return train_xs, train_ys, test_xs, test_ys, train_indices, test_indices


train_xs, train_ys, test_xs, test_ys, train_idx, test_idx = train_test_split(
    banknote_xs,
    banknote_ys,
    test_size=0.2,
    seed=42,
)

print("train:", len(train_xs))
print("test:", len(test_xs))
print("index overlap:", len(set(train_idx).intersection(set(test_idx))))

feature_overlap = set(tuple(x) for x in train_xs).intersection(
    set(tuple(x) for x in test_xs)
)
print("feature overlap:", len(feature_overlap))

scaler = StandardScaler()

train_xs_scaled = scaler.fit_transform(train_xs).tolist()
test_xs_scaled = scaler.transform(test_xs).tolist()

In [ ]:
#make batches for training the model in mini-batch gradient descent
def make_batches(xs, ys, batch_size=32, seed=None):
    indices = list(range(len(xs)))

    if seed is not None:
        random.seed(seed)

    random.shuffle(indices)

    for start in range(0, len(indices), batch_size):
        batch_indices = indices[start:start + batch_size]

        batch_xs = [xs[i] for i in batch_indices]
        batch_ys = [ys[i] for i in batch_indices]

        yield batch_xs, batch_ys

In [ ]:
random.seed(42)

model = BinaryMLPClassifier(4, [8, 4], activation="tanh")
optimizer = Adam(model.parameters(), lr=0.01)

banknote_history = {
    "train_loss": [],
    "train_accuracy": [],
    "test_accuracy": [],
}

for epoch in range(31):
    batch_losses = []

    for batch_xs, batch_ys in make_batches(
        train_xs_scaled,
        train_ys,
        batch_size=32,
        seed=epoch,
    ):
        loss = model.loss(batch_xs, batch_ys)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_losses.append(loss.data)

    avg_loss = sum(batch_losses) / len(batch_losses)
    train_acc = model.accuracy(train_xs_scaled, train_ys)
    test_acc = model.accuracy(test_xs_scaled, test_ys)

    banknote_history["train_loss"].append(avg_loss)
    banknote_history["train_accuracy"].append(train_acc)
    banknote_history["test_accuracy"].append(test_acc)

    if epoch % 5 == 0 :
        print(
            epoch,
            "loss:", round(avg_loss, 4),
            "train_acc:", round(train_acc, 4),
            "test_acc:", round(test_acc, 4),
        )

In [ ]:
plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(banknote_history["train_loss"])
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Banknote Adam Train Loss")

plt.subplot(1, 2, 2)
plt.plot(banknote_history["train_accuracy"], label="train")
plt.plot(banknote_history["test_accuracy"], label="test")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Banknote Adam Accuracy")
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
def make_noisy_xor(repeats_per_point=100, noise_rate=0.25, seed=42):
    random.seed(seed)
    base_xs = [
        [0.0, 0.0],
        [0.0, 1.0],
        [1.0, 0.0],
        [1.0, 1.0],
    ]
    base_ys = [0.0, 1.0, 1.0, 0.0]

    xs = []
    ys = []
    for x, y in zip(base_xs, base_ys):
        for _ in range(repeats_per_point):
            noisy_y = y
            if random.random() < noise_rate:
                noisy_y = 1.0 - y
            xs.append(x)
            ys.append(noisy_y)
    return xs, ys

noisy_xor_xs, noisy_xor_ys = make_noisy_xor(
    repeats_per_point=100,
    noise_rate=0.25,
    seed=42,
)

In [ ]:
random.seed(42)

model = BinaryMLPClassifier(2, [4, 4], activation="tanh")
optimizer = Adam(model.parameters(), lr=0.03)

for epoch in range(100):
    loss = model.loss(noisy_xor_xs, noisy_xor_ys)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        acc = model.accuracy(noisy_xor_xs, noisy_xor_ys)
        print(epoch, round(loss.data, 4), round(acc, 4))
        